# Benchmark on the test strip

Scores stock Point-SAM and our fine-tune on whole test tiles: ground removal, clicks, masks, merging, then F1 at IoU 0.5. Each result is shown for three click sources, because low automatic-click F1 can come from the clicks, the masks or both:

- **grid, stems:** automatic clicks, which is what real use would see.
- **perfect:** one click per tree from the true labels, which shows how good the masks can be.

**Two scoring rules:** `strict` counts every unmatched prediction as wrong. `ignore_non_tree` skips predictions mostly on non-tree points, since some of those are real understory without a label.

**Three measures, because each answers a different question:**

- `f1`: how many trees were correctly separated, counting a match at IoU 0.5 or above.
- `mean_tree_iou`: each real tree's best IoU with any prediction, averaged over all trees. A missed tree counts as 0.
- `mean_matched_iou`: mask quality for matched trees only. It ignores missed trees, so read it alongside the other two.

**Run this once, after choosing the checkpoint.** Tuning anything after seeing these numbers would spoil the test set. Needs a GPU.

In [ ]:
import os
import sys

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, "src")

import numpy as np
import pandas as pd
import yaml
from IPython.display import Image, display

from treeseg.engine.benchmark import prepare_tile, run_benchmark, summarise
from treeseg.utils.curves import plot_f1_bars
from treeseg.utils.view3d import plot_compare_3d

cfg = yaml.safe_load(open(os.environ.get("TREESEG_CONFIG", "configs/experiment/benchmark.yaml")))
cfg

## Run

Rows are saved to `results.csv` as they finish, and each prediction is saved as `.npy`.

In [ ]:
results_path = run_benchmark(cfg)

## Results

In [ ]:
results_path = os.path.join(cfg["out_dir"], "results.csv")
summary = summarise(results_path)
summary.to_csv(os.path.join(cfg["out_dir"], "summary.csv"), index=False)
summary

In [ ]:
figure_path = os.path.join(cfg["out_dir"], "f1_bars.png")
plot_f1_bars(summary, figure_path)
display(Image(figure_path))

### Per tile

Averages can hide one tile doing all the work.

In [ ]:
per_tile = pd.read_csv(results_path)
per_tile[per_tile["rule"] == "ignore_non_tree"][["checkpoint", "tile", "clicks_from", "f1", "mean_tree_iou", "mean_matched_iou", "precision", "recall", "trees", "predictions", "click_recall"]].round(3)

## See it

True trees next to predicted trees. Colours are separate per panel, so the same tree has different colours on each side. Change the three settings to look at other results.

In [ ]:
CHECKPOINT, TILE, CLICKS = "isprs_lora", cfg["tiles"][0], "perfect"

tile = prepare_tile(cfg["tile_dir"], TILE, cfg["csf"], cfg["min_tree_points"])
pred = np.load(os.path.join(cfg["out_dir"], f"{CHECKPOINT}_{TILE}_{CLICKS}_pred.npy"))
plot_compare_3d(tile["xyz"], tile["ids"], pred, f"{CHECKPOINT}, {TILE}, {CLICKS} clicks")